In [1]:
import random
import pickle

In [2]:
class TicTacToe:
    def __init__(self):
        self.board = [' '] * 9

    def display(self):
        for i in range(0, 9, 3):
            print(" | ".join(self.board[i:i+3]))
            if i < 6:
                print("---------")

    def available_moves(self):
        return [i for i, x in enumerate(self.board) if x == ' ']

    def make_move(self, pos, player):
        if pos in self.available_moves():
            self.board[pos] = player
            return True
        return False

    def check_winner(self):
        wins = [(0,1,2), (3,4,5), (6,7,8),
                (0,3,6), (1,4,7), (2,5,8),
                (0,4,8), (2,4,6)]

        for a,b,c in wins:
            if self.board[a] == self.board[b] == self.board[c] != ' ':
                return self.board[a]

        return 'Draw' if not self.available_moves() else None

In [3]:
class QLearningAgent:
    def __init__(self):
        self.q = {}
        self.lr = 0.1
        self.discount = 0.9
        self.epsilon = 0.2

    def get_state(self, board, player):
        return tuple('X' if x == player else 'O' if x != ' ' else ' '
                     for x in board)

    def choose_action(self, board, player):
        state = self.get_state(board, player)
        moves = [i for i, x in enumerate(board) if x == ' ']

        if random.random() < self.epsilon:
            return random.choice(moves)

        return max(moves, key=lambda a: self.q.get((state, a), 0))

    def update(self, state, action, reward, next_state, done):
        old_q = self.q.get((state, action), 0)

        future = 0 if done else max(
            [self.q.get((next_state, a), 0)
             for a in range(9) if next_state[a] == ' '] or [0]
        )

        self.q[(state, action)] = old_q + self.lr * (
            reward + self.discount * future - old_q
        )

In [4]:
agent = QLearningAgent()

for episode in range(100000):
    game = TicTacToe()
    player = 'X'
    previous = None

    while True:
        state = agent.get_state(game.board, player)
        action = agent.choose_action(game.board, player)

        game.make_move(action, player)
        result = game.check_winner()

        if result:
            reward = 1 if result == player else 0
            agent.update(state, action, reward, None, True)

            if previous:
                ps, pa, pp = previous
                pr = -1 if result == player else 0
                agent.update(ps, pa, pr, None, True)
            break

        if previous:
            ps, pa, pp = previous
            next_state = agent.get_state(game.board, pp)
            agent.update(ps, pa, 0, next_state, False)

        previous = (state, action, player)
        player = 'O' if player == 'X' else 'X'

    agent.epsilon = max(0.05, agent.epsilon * 0.99995)

print("Training completed!")
print("Total Episodes: 100000")
print("Q-Table Size:", len(agent.q))

Training completed!
Total Episodes: 100000
Q-Table Size: 7526


In [5]:
agent.epsilon = 0

wins = draws = losses = 0

for _ in range(1000):
    game = TicTacToe()

    while True:
        action = agent.choose_action(game.board, 'X')
        game.make_move(action, 'X')
        result = game.check_winner()

        if result:
            wins += result == 'X'
            draws += result == 'Draw'
            losses += result == 'O'
            break

        action = random.choice(game.available_moves())
        game.make_move(action, 'O')
        result = game.check_winner()

        if result:
            wins += result == 'X'
            draws += result == 'Draw'
            losses += result == 'O'
            break

print("Wins:", wins)
print("Draws:", draws)
print("Losses:", losses)
print("Win Rate:", wins / 1000 * 100, "%")

Wins: 955
Draws: 45
Losses: 0
Win Rate: 95.5 %


In [ ]:
game = TicTacToe()

while True:
    game.display()
    pos = int(input("Enter position (0-8): "))

    if not game.make_move(pos, 'X'):
        print("Invalid move!")
        continue

    result = game.check_winner()
    if result:
        game.display()
        print("You win!" if result == 'X' else "Draw!")
        break

    action = agent.choose_action(game.board, 'O')
    game.make_move(action, 'O')

    print("Computer played:", action)
    result = game.check_winner()

    if result:
        game.display()
        print("Computer wins!" if result == 'O' else "Draw!")
        break

  |   |  
---------
  |   |  
---------
  |   |  
Computer played: 4
  |   | X
---------
  | O |  
---------
  |   |  
Computer played: 0
O |   | X
---------
X | O |  
---------
  |   |  
Invalid move!
O |   | X
---------
X | O |  
---------
  |   |  
Computer played: 5
O |   | X
---------
X | O | O
---------
  |   | X
Computer played: 6
O | X | X
---------
X | O | O
---------
O |   | X
